# Car Price Prediction

End-to-end training notebook. Design decisions:

- **All preprocessing lives inside one sklearn `Pipeline`** — split first, fit only on train, zero leakage, one deployable artifact.
- **Feature set = the app's input contract** (`Brand, Model, Year, Fuel_Type, Transmission, Owner_Type, Kms_Driven`), imported from `preprocessing.py` so the model and API can never diverge.
- **Target trained as `log1p(Price)`** via `TransformedTargetRegressor`, so predictions and metrics below are in actual currency units.
- Production training is scripted in `train.py`; this notebook is for EDA and model selection.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from preprocessing import CATEGORICAL, FEATURES, NUMERIC, TARGET, clean

df_raw = pd.read_csv('car_price_prediction.csv', usecols=FEATURES + [TARGET])
df_raw = df_raw.drop_duplicates().reset_index(drop=True)
print(df_raw.shape)
df_raw.head()

## Data quality

`Fuel_Type` contains case/whitespace variants and typos (`'PETROL'`, `' Diesel'`, `'electrik'`, `'hybridd'`). `clean()` maps them to 5 canonical labels. Missing `Fuel_Type`/`Transmission` are left as NaN — imputation is the pipeline's job, fitted on train only.

In [ ]:
print('Raw fuel labels:', sorted(df_raw['Fuel_Type'].dropna().unique()))
df = clean(df_raw)
print('Clean fuel labels:', sorted(df['Fuel_Type'].dropna().unique()))
df.isna().sum()

In [ ]:
# Price is right-skewed; log1p makes it near-normal -> train on log scale.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df[TARGET], bins=50, ax=axes[0]).set_title('Price')
sns.histplot(np.log1p(df[TARGET]), bins=50, ax=axes[1]).set_title('log1p(Price)')
plt.tight_layout()
plt.show()

## Split, then model comparison

Each candidate is a full pipeline (impute + encode + regress on log target). Metrics are computed on the untouched test split, in currency units.

In [ ]:
from sklearn.model_selection import train_test_split

X, y = df[FEATURES], df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

In [ ]:
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.tree import DecisionTreeRegressor

from train import build_pipeline  # HistGradientBoosting, the production candidate

ohe_prep = ColumnTransformer([
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('ohe', OneHotEncoder(handle_unknown='ignore', min_frequency=20))]), CATEGORICAL),
    ('num', 'passthrough', NUMERIC),
])
ord_prep = ColumnTransformer([
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1,
                           encoded_missing_value=-1), CATEGORICAL),
    ('num', 'passthrough', NUMERIC),
])

def log_target(reg):
    return TransformedTargetRegressor(regressor=reg, func=np.log1p, inverse_func=np.expm1)

candidates = {
    'Ridge': Pipeline([('prep', ohe_prep), ('model', log_target(Ridge()))]),
    'DecisionTree': Pipeline([('prep', ord_prep),
                              ('model', log_target(DecisionTreeRegressor(max_depth=12, min_samples_leaf=20, random_state=42)))]),
    'HistGradientBoosting': build_pipeline(random_state=42),
}

rows, fitted = [], {}
for name, pipe in candidates.items():
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    rows.append({'model': name,
                 'MAE': mean_absolute_error(y_test, pred),
                 'RMSE': mean_squared_error(y_test, pred) ** 0.5,
                 'R2': r2_score(y_test, pred)})
    fitted[name] = pipe

results = pd.DataFrame(rows).sort_values('R2', ascending=False).reset_index(drop=True)
results

## Persist one artifact

The full pipeline (imputation + encoding + model + target inversion) is the only file the app needs. No separate encoder pickles.

In [ ]:
import json
from pathlib import Path

import joblib

best_name = results.iloc[0]['model']
Path('model').mkdir(exist_ok=True)
joblib.dump(fitted[best_name], 'model/car_price_pipeline.pkl', compress=3)
Path('model/metrics.json').write_text(json.dumps(results.iloc[0].to_dict(), indent=2))
print('saved:', best_name)

In [ ]:
# Inference smoke test: load the artifact, predict on raw rows (no target column).
loaded = joblib.load('model/car_price_pipeline.pkl')
sample = X_test.head(5)
pd.DataFrame({'actual': y_test.head(5).round(0).values,
              'predicted': loaded.predict(sample).round(0)})